# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and analyze data from the FAIR^2 dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library. 

### Dataset Source
The dataset is defined via a Croissant schema available at the following URL:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Ensure mlcroissant is installed in the environment
!pip install -q mlcroissant

## 1. Data Loading
Load the dataset metadata and retrieve information about available record sets using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)

# Show dataset metadata: title and description
md = dataset.metadata
print("\033[1mDataset Title:\033[0m", md.name)
print("\033[1mDescription:\033[0m", md.description)


## 2. Data Overview
Inspect available record sets, their `@id`s, and contained fields and columns. This information is essential for referencing entities throughout your workflow.

In [ ]:
# List all record sets and their fields by @id

record_sets = dataset.record_sets
print("Found record sets:")
for rs in record_sets:
    print(f"  RecordSet @id: {rs['@id']}")
    if 'field' in rs:
        print("    Fields:")
        for field in rs['field']:
            if isinstance(field, dict):
                print(f"      • {field['@id']}")
            else:
                print(f"      • {field}")
    if 'column' in rs:
        print("    Columns:")
        for col in rs['column']:
            if isinstance(col, dict):
                print(f"      - {col['@id']}")
            else:
                print(f"      - {col}")
    print()
# Save list of record set @ids for subsequent steps
record_set_ids = [rs['@id'] for rs in record_sets]
print("All RecordSet @ids:", record_set_ids)


## 3. Data Extraction
Load data from each record set into a Pandas DataFrame. Reference record sets and fields by their `@id` (see the overview above).


In [ ]:
# Data extraction: load all available record sets into DataFrames.
dataframes = {}
for record_set_id in record_set_ids:
    print(f"Loading records for RecordSet @id: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"  → Loaded {len(df)} rows, {len(df.columns)} columns.\n")
# For demonstration, pick the first record set (if available)
if record_set_ids:
    main_record_set_id = record_set_ids[0]
    print(f"Main RecordSet @id: {main_record_set_id}")
    print("Columns in this data:", dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())
else:
    print("No record sets found.")

## 4. Exploratory Data Analysis (EDA)
We'll perform simple exploratory steps. Replace `<numeric_field_id>` and `<group_field>` with appropriate `@id`s from your dataset. 

For illustration, this cell auto-detects a numeric field and a group field (categorical) from the columns (by heuristics).

In [ ]:
import numpy as np

# Pick main DataFrame
df = dataframes.get(main_record_set_id)

if df is not None and not df.empty:
    # Heuristically find a likely numeric field
    numeric_field = None
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field = col
            break
    if not numeric_field:
        for col in df.columns:
            # Try to coerce to numeric and check for reasonable non-NaN content
            coerced = pd.to_numeric(df[col], errors='coerce')
            if not coerced.isnull().all():
                numeric_field = col
                df[col] = coerced
                break
    
    if numeric_field:
        print(f"Numeric field chosen: {numeric_field}")
        threshold = np.nanmean(df[numeric_field])
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records with {numeric_field} > {threshold:.2f} (mean):")
        display(filtered_df.head())
    
        # Normalize
        filtered_df[f"{numeric_field}_normalized"] = (
            filtered_df[numeric_field] - filtered_df[numeric_field].mean()
        ) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
        
        # Try to find a group field to group by (categorical, not numeric)
        group_field = None
        for col in df.columns:
            if col != numeric_field and df[col].nunique() < len(df)/2:
                group_field = col
                break
        if group_field:
            print(f"Grouping by field: {group_field}")
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
            print(f"Mean {numeric_field} by {group_field}:")
            display(grouped_df.head())
        else:
            print("No suitable group field found for grouping.")
    else:
        print("No numeric fields found in the dataset.")
else:
    print("No data loaded for the main record set.")

## 5. Visualization
Visualize the distribution of the chosen numeric field and, if grouping is possible, mean values per group.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import seaborn as sns

if df is not None and not df.empty and numeric_field:
    plt.figure(figsize=(7, 4))
    sns.histplot(df[numeric_field].dropna(), bins=20, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.show()
    
    if 'group_field' in locals() and group_field:
        plt.figure(figsize=(10, 5))
        group_means = df.groupby(group_field)[numeric_field].mean()
        group_means.plot(kind='bar')
        plt.ylabel(f"Mean {numeric_field}")
        plt.title(f"Mean {numeric_field} by {group_field}")
        plt.show()
else:
    print("No numeric fields available for visualization.")

## 6. Conclusion
This notebook demonstrated how to load, examine, and perform basic analysis on a FAIR^2 Croissant dataset using `mlcroissant`. Using the `@id` fields from the schema ensures precise referencing for all entities. Continue your analysis based on your research questions or domain knowledge.
